In [ ]:
#@title 1. Setup - run once { display-mode: "form" }
#@markdown **Runtime > Run all** (if Colab warns the notebook is not by Google, click **Run anyway**).
#@markdown Cell 2 then prints the public API link, its docs and the key. Keep this tab open.
#@markdown Tick to keep downloads in your Google Drive (asks you to sign in).
#@markdown Unticked, they go to `/content/downloads`, wiped when the runtime ends.
use_drive = False  #@param {type:"boolean"}
drive_folder = "yt-downloads"  #@param {type:"string"}
#@markdown Leave blank to keep the saved key (one is generated the first time).
api_key = ""  #@param {type:"string"}

import os, subprocess

REPO, DIR = "https://github.com/freelancermeer/ytmeer.git", "/content/ytmeer"

!apt-get -qq install -y ffmpeg aria2 > /dev/null 2>&1
!pip -q install "yt-dlp>=2026.7.4" "curl_cffi>=0.10,<0.16" "gradio>=6,<7"

if os.path.isdir(f"{DIR}/.git"):
    subprocess.run(["git", "-C", DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, DIR], check=True)

if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = f"/content/drive/MyDrive/{drive_folder.strip().strip('/') or 'yt-downloads'}"
else:
    OUTDIR = "/content/downloads"
OUTDIR = os.path.abspath(OUTDIR)
os.makedirs(OUTDIR, exist_ok=True)

# Saved beside the code, not in the environment, so cell 2 - and any server it
# starts - still has them after a runtime restart.
with open(f"{DIR}/.api_outdir", "w") as f:
    f.write(OUTDIR)
if api_key.strip():
    # Pending until cell 2 next starts a server, so a server still downloading
    # stays reachable with the key it was started with.
    fd = os.open(f"{DIR}/.api_key_new", os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with os.fdopen(fd, "w") as f:
        f.write(api_key.strip() + "\n")
print("Setup done. Downloads go to", OUTDIR)

In [ ]:
#@title 2. Start the API { display-mode: "form" }
#@markdown Tick to replace the server even if nothing changed - any jobs it is running are stopped first.
force_restart = False  #@param {type:"boolean"}
#@markdown Keep this cell running, so Colab does not shut an idle runtime down. Stopping the cell leaves the API running.
keep_running = True  #@param {type:"boolean"}

import hashlib, json, os, shutil, signal, socket, subprocess, sys, time, urllib.error, urllib.request

DIR = "/content/ytmeer"


def _read(name):
    try:
        with open(f"{DIR}/{name}") as f:
            return f.read().strip()
    except OSError:
        return ""


def _get(port, path):
    """The JSON the API on this port answers, or None."""
    req = urllib.request.Request(f"http://127.0.0.1:{port}/api/{path}",
                                 headers={"X-API-Key": _read(".api_key")})
    try:
        with urllib.request.urlopen(req, timeout=3) as r:
            return json.load(r)
    except Exception:
        return None


def _reachable(public, tries):
    """True if the public link answers, else the last error. Checked the way outside
    code reaches it: out through gradio.live and back."""
    result = None
    for attempt in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(
                    f"{public}/api/health", headers={"X-API-Key": _read(".api_key")}),
                    timeout=10) as r:
                return r.status == 200
        except Exception as e:
            result = e
            if attempt < tries - 1:
                time.sleep(2)
    return result


FOLDER = _read(".api_outdir")
if not FOLDER:
    raise RuntimeError("Run cell 1 first - it chooses the download folder.")
PORT = int(_read(".api_port") or 8000)

# A cookies.txt uploaded with the Files panel lands in /content. Every job, and the
# YouTube check below, looks for it in the download folder - so copy it there, owner-
# only, whenever the uploaded one is newer. No restart needed: each job checks.
DROP = "/content/cookies.txt"
if os.path.isfile(DROP):
    shared = f"{FOLDER}/cookies.txt"
    if not os.path.isfile(shared) or os.path.getmtime(DROP) > os.path.getmtime(shared):
        os.makedirs(FOLDER, exist_ok=True)
        with open(DROP, "rb") as src:
            data = src.read()
        fd = os.open(shared, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
        with os.fdopen(fd, "wb") as dst:
            dst.write(data)
        print(f"  cookies.txt from the Files panel copied to {shared}")
code = hashlib.sha1()
for name in ("api.py", "downloader.py"):
    with open(f"{DIR}/{name}", "rb") as f:
        code.update(f.read())
code = code.hexdigest()[:12]

new_key = _read(".api_key_new")
if new_key and new_key == _read(".api_key"):
    os.remove(f"{DIR}/.api_key_new")
    new_key = ""


def _running(what):
    return subprocess.run(["pgrep", "-f", f"{DIR}/{what}[.]py"], capture_output=True, text=True).stdout.split()


ping, health = _get(PORT, "ping"), _get(PORT, "health")
# A server that is alive but does not answer counts as busy, not as gone.
busy = ping.get("busy") if ping else bool(_running("api"))
current = (bool(health) and health["code"] == code and health["default_folder"] == FOLDER
           and not new_key and not force_restart)
if current and _reachable(health.get("public_url"), 2) is not True:
    current = False                       # no public link, or it has stopped answering

if not current and busy and not force_restart:
    # Never throw away a download in progress because cell 1 changed something.
    print(f"NOTE: the API on port {PORT} is still downloading, so it was left as it is;\n"
          "      cell 1's changes apply once it is replaced. Re-run this cell when its\n"
          "      jobs are done, or tick force_restart.")
    if not health:
        raise RuntimeError("...and it cannot be reached. Tick force_restart to replace it.")
elif not current:
    # Stop the server first - it stops its own jobs as it shuts down - then any
    # downloader still going, left by a server that died without doing so. Each
    # downloader leads its own process group, so that reaches its yt-dlp too.
    subprocess.run(["pkill", "-f", f"{DIR}/api[.]py"])
    for _ in range(120):
        if not _running("api"):
            break
        time.sleep(0.25)
    for pid in _running("downloader"):
        try:
            os.killpg(int(pid), signal.SIGINT)
        except OSError:
            pass
    for _ in range(120):
        if not _running("downloader"):
            break
        time.sleep(0.25)
    if new_key:
        os.replace(f"{DIR}/.api_key_new", f"{DIR}/.api_key")
    with socket.socket() as s:            # the port may belong to something else
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)   # as uvicorn binds
        try:
            s.bind(("127.0.0.1", PORT))
        except OSError:
            s.bind(("127.0.0.1", 0))
            PORT = s.getsockname()[1]
    with open(f"{DIR}/.api_port", "w") as f:
        f.write(str(PORT))
    env = {k: v for k, v in os.environ.items() if not k.startswith("YTDL_")}
    with open(f"{DIR}/api.log", "a") as log:
        subprocess.Popen([sys.executable, f"{DIR}/api.py", "--port", str(PORT), "--outdir", FOLDER,
                          "--share"],
                         stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(240):                  # the public link takes a few seconds to open
        if _get(PORT, "health"):
            break
        time.sleep(0.5)
    else:
        with open(f"{DIR}/api.log") as log:
            print(log.read()[-3000:])
        raise RuntimeError("The API did not start - its log is above.")

API_KEY = _read(".api_key")
API_URL = f"http://127.0.0.1:{PORT}/api"
HEADERS = {"X-API-Key": API_KEY}
info = _get(PORT, "health") or {}
PUBLIC_URL = f"{info['public_url']}/api" if info.get("public_url") else None

reachable = None
if PUBLIC_URL:
    print("  checking the public link...")
    reachable = _reachable(info["public_url"], 5)

if PUBLIC_URL:
    state = ("reachable from the internet" if reachable is True
             else f"NOT reachable: {reachable} - re-run this cell")
    print(f"  public  {PUBLIC_URL}      ({state})")
    print(f"  docs    {PUBLIC_URL}/docs      (open it, click Authorize, paste the key)")
else:
    print(f"  public  none - Gradio could not open a share link; re-run this cell ({DIR}/api.log)")
print(f"  local   {API_URL}      (for cells in this notebook)")
print(f"  key     {API_KEY}")
print(f"  folder  {info.get('default_folder', FOLDER)}")
# Can this runtime download at all? YouTube blocks some Colab IPs, and rejects cookies
# that were rotated after export. Try what the downloader tries - cookies (if any),
# then without them; the default client, then mweb - and say which one worked.
cookies = f"{info.get('default_folder', FOLDER)}/cookies.txt"
has_cookies = os.path.exists(cookies)
youtube, seen = None, []
for with_cookies in ([True, False] if has_cookies else [False]):
    for client in ([], ["--extractor-args", "youtube:player_client=mweb,default"]):
        try:
            probe = subprocess.run(["yt-dlp", "--skip-download", "--no-warnings", "--print", "id", *client,
                                    *(["--cookies", shutil.copyfile(cookies, "/tmp/ytdl_probe_cookies.txt")]
                                      if with_cookies else []),
                                    "https://www.youtube.com/watch?v=jNQXAC9IVRw"],
                                   capture_output=True, text=True, timeout=120)
        except Exception as e:
            seen.append(str(e))
            continue
        if probe.returncode == 0:
            if with_cookies or not has_cookies:
                youtube = "OK - this runtime can download" + (" (with your cookies)" if with_cookies else "")
            elif "needs to be reloaded" in " ".join(seen).lower():
                youtube = ("OK without cookies - YouTube rejects your cookies.txt, so jobs\n"
                           "          carry on without it (re-export it if you ever need it)")
            else:
                youtube = f"OK without cookies (the check with your cookies.txt failed: {seen[-1][:120]})"
            break
        seen.append((probe.stderr.strip().splitlines() or ["no output"])[-1])
    if youtube:
        break
if os.path.exists("/tmp/ytdl_probe_cookies.txt"):
    os.remove("/tmp/ytdl_probe_cookies.txt")
if not youtube:
    low = " ".join(seen).lower()
    if "not a bot" in low or "needs to be reloaded" in low:
        youtube = ("BLOCKED - YouTube refuses this runtime" +
                   (" and rejects your cookies.txt" if has_cookies and "reloaded" in low else "") + ".\n"
                   "          Fix: Runtime > Disconnect and delete runtime, then Run all (a new IP),\n"
                   "          or drag a freshly exported cookies.txt into the Files panel and run this cell.")
    else:
        youtube = f"check failed: {seen[-1][:200] if seen else 'no output'}"
print(f"  cookies {'yes' if os.path.exists(cookies) else 'none (only needed if YouTube blocks)'}")
print(f"  youtube {youtube}")
print("\n  Keep this tab open: the link and the server stop when the runtime disconnects.")
print("  Then Run all again - the public URL will be a new one.")

CODE = '''import requests, time

BASE = __BASE__
H = {"X-API-Key": __KEY__}

def call(method, path, **kw):
    r = requests.request(method, BASE + path, headers=H, timeout=60, **kw)
    if not r.ok:
        raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
    return r.json()

job = call("POST", "/jobs", json={"links": ["https://www.youtube.com/@SomeChannel"],
                                  "views": 1000, "limit": 3, "days": 5})
since = 0
while True:                           # each video arrives as soon as it has finished
    page = call("GET", f"/jobs/{job['job_id']}/videos", params={"since": since})
    for v in page["videos"]:
        if v["status"] in ("ok", "skipped"):
            print(v["title"], v["quality"])
            if "transcript" in v["downloads"]:   # any file, fetched over the API from anywhere
                text = requests.get(BASE + v["downloads"]["transcript"], headers=H).text
            # a video:  requests.get(BASE + v["downloads"]["video"], headers=H, stream=True)
    since = page["next"]
    if page["done"]:
        break
    time.sleep(5)
job = call("GET", f"/jobs/{job['job_id']}")
print(job["state"], job["error"], job["failures"])
'''
if PUBLIC_URL:
    print("\nGive your other code this, with your URL and key already in it:")
else:
    print("\nNo public link, so only a cell of this notebook can reach the API:")
print("----- copy from here -----")
print(CODE.replace("__BASE__", json.dumps(PUBLIC_URL or API_URL))
          .replace("__KEY__", json.dumps(API_KEY)), end="")
print("----- to here -----")
print("Each video lists its files twice: \"files\" are paths on this Colab machine (for a\n"
      "cell of this notebook), \"downloads\" are API paths that fetch them from anywhere.")

if keep_running:
    print("\nThis cell keeps running so Colab keeps the runtime. Stop it any time - the API carries on.")
    try:
        while True:
            try:
                with urllib.request.urlopen(f"{API_URL}/ping", timeout=10) as r:
                    state = "downloading" if json.load(r).get("busy") else "idle"
            except Exception:
                state = "NOT ANSWERING - run this cell again"
            print(f"\r  {time.strftime('%H:%M:%S')}  API {state}            ", end="", flush=True)
            time.sleep(60)
    except KeyboardInterrupt:
        print("\nStopped watching; the API is still running.")
